# PyTorch Deep Learning with cachepy

This notebook shows how cachepy accelerates a typical PyTorch workflow by caching:
- **Data preprocessing** — download, normalize, augment
- **Feature extraction** — forward pass through a pretrained model
- **Training runs** — cache trained models for different hyperparameters
- **Evaluation** — avoid recomputing metrics when only visualization changes

We use MNIST for simplicity, but the same pattern applies to any PyTorch pipeline.

In [ ]:
import sys, time, shutil, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms

sys.path.insert(0, str(Path.cwd().parent.parent))
from cachepy import cache_file, cache_tree_nodes, cache_tree_reset
from cachepy.cache_file import cache_stats, _file_state_cache

warnings.filterwarnings("ignore")

CACHE_DIR = Path("pytorch_cache")
DATA_DIR = Path("data")

def fresh_start():
    if CACHE_DIR.exists():
        shutil.rmtree(CACHE_DIR)
    cache_tree_reset()
    _file_state_cache.clear()

fresh_start()

import logging
logging.basicConfig(level=logging.INFO, format="[cachepy] %(message)s", force=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Cache: {CACHE_DIR.resolve()}")

## 1. Data Loading & Preprocessing

Download MNIST and convert to tensors. Caching this step avoids repeated downloads and preprocessing.

In [ ]:
@cache_file(CACHE_DIR, backend="pickle", verbose=True)
def load_mnist(data_dir="data", flatten=False):
    """Download MNIST and return train/test tensors."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    train_ds = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(data_dir, train=False, download=True, transform=transform)

    # Stack into single tensors
    X_train = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
    y_train = torch.tensor([train_ds[i][1] for i in range(len(train_ds))])
    X_test = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
    y_test = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

    if flatten:
        X_train = X_train.view(X_train.size(0), -1)
        X_test = X_test.view(X_test.size(0), -1)

    return {"X_train": X_train, "y_train": y_train,
            "X_test": X_test, "y_test": y_test}

t0 = time.time()
data = load_mnist(str(DATA_DIR), flatten=False)
print(f"Loaded in {time.time()-t0:.2f}s")
print(f"Train: {data['X_train'].shape}, Test: {data['X_test'].shape}")

## 2. Model Definition

A simple CNN for digit classification.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_filters=32, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, n_filters, 3, padding=1)
        self.conv2 = nn.Conv2d(n_filters, n_filters * 2, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(n_filters * 2 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = SimpleCNN()
n_params = sum(p.numel() for p in model.parameters())
print(f"SimpleCNN: {n_params:,} parameters")

## 3. Cached Training

The training function is wrapped with `@cache_file`. Different hyperparameter combinations produce different cache entries. Re-running the notebook with the same hyperparameters loads the trained model instantly.

In [ ]:
@cache_file(CACHE_DIR, backend="pickle", verbose=True)
def train_model(X_train, y_train, n_filters=32, dropout=0.25,
                lr=1e-3, epochs=3, batch_size=128, seed=42):
    """Train a CNN and return weights + training history."""
    torch.manual_seed(seed)

    model = SimpleCNN(n_filters=n_filters, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(X_train.to(DEVICE), y_train.to(DEVICE))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = {"loss": [], "acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_X.size(0)
            correct += (output.argmax(1) == batch_y).sum().item()
            total += batch_X.size(0)

        epoch_loss = total_loss / total
        epoch_acc = correct / total
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)
        print(f"  Epoch {epoch+1}/{epochs}: loss={epoch_loss:.4f}, acc={epoch_acc:.4f}")

    # Return state dict (serializable) + history
    return {"state_dict": model.cpu().state_dict(), "history": history,
            "n_filters": n_filters, "dropout": dropout}

t0 = time.time()
result = train_model(data["X_train"], data["y_train"],
                     n_filters=32, dropout=0.25, lr=1e-3, epochs=3)
print(f"\nTraining done in {time.time()-t0:.2f}s")
print(f"Final accuracy: {result['history']['acc'][-1]:.4f}")

## 4. Cached Evaluation

Evaluation is cached separately. You can change visualization code without re-running inference.

In [ ]:
@cache_file(CACHE_DIR, backend="pickle", verbose=True)
def evaluate_model(train_result, X_test, y_test):
    """Evaluate a trained model and return predictions + metrics."""
    model = SimpleCNN(
        n_filters=train_result["n_filters"],
        dropout=train_result["dropout"]
    )
    model.load_state_dict(train_result["state_dict"])
    model.eval()

    with torch.no_grad():
        logits = model(X_test)
        preds = logits.argmax(1)
        probs = F.softmax(logits, dim=1)

    acc = (preds == y_test).float().mean().item()

    # Per-class accuracy
    class_acc = {}
    for c in range(10):
        mask = y_test == c
        class_acc[c] = (preds[mask] == y_test[mask]).float().mean().item()

    # Confusion matrix
    confusion = torch.zeros(10, 10, dtype=torch.long)
    for t, p in zip(y_test, preds):
        confusion[t, p] += 1

    return {"accuracy": acc, "class_acc": class_acc,
            "confusion": confusion, "preds": preds, "probs": probs}

t0 = time.time()
metrics = evaluate_model(result, data["X_test"], data["y_test"])
print(f"Evaluated in {time.time()-t0:.2f}s")
print(f"Test accuracy: {metrics['accuracy']:.4f}")
print(f"\nPer-class accuracy:")
for c, acc in metrics["class_acc"].items():
    print(f"  Digit {c}: {acc:.4f}")

## 5. Visualization (not cached)

Plotting code changes frequently — no need to cache it. The expensive evaluation above is cached.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Training curves
ax = axes[0]
epochs = range(1, len(result["history"]["loss"]) + 1)
ax.plot(epochs, result["history"]["loss"], "o-", color="#e74c3c", label="Loss")
ax2 = ax.twinx()
ax2.plot(epochs, result["history"]["acc"], "s-", color="#2ecc71", label="Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss", color="#e74c3c")
ax2.set_ylabel("Accuracy", color="#2ecc71")
ax.set_title("Training Curves")

# Per-class accuracy
ax = axes[1]
digits = list(metrics["class_acc"].keys())
accs = list(metrics["class_acc"].values())
colors = ["#e74c3c" if a < 0.97 else "#2ecc71" for a in accs]
ax.bar(digits, accs, color=colors)
ax.set_xlabel("Digit")
ax.set_ylabel("Accuracy")
ax.set_title(f"Per-Class Accuracy (overall: {metrics['accuracy']:.3f})")
ax.set_ylim(0.9, 1.0)
ax.set_xticks(digits)

# Confusion matrix
ax = axes[2]
im = ax.imshow(metrics["confusion"].numpy(), cmap="Blues")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
plt.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
plt.show()

## 6. Hyperparameter Search with Caching

The key benefit of cachepy for deep learning: iterate on hyperparameters without re-training completed configs. If you add a new learning rate to the grid, only the new ones train.

In [ ]:
# Use a smaller subset for the hyperparameter search demo
X_sub = data["X_train"][:5000]
y_sub = data["y_train"][:5000]

configs = [
    {"n_filters": 16, "lr": 1e-3, "epochs": 3},
    {"n_filters": 32, "lr": 1e-3, "epochs": 3},
    {"n_filters": 32, "lr": 5e-4, "epochs": 3},
    {"n_filters": 64, "lr": 1e-3, "epochs": 3},
]

hp_results = []
for i, cfg in enumerate(configs):
    print(f"\nConfig {i+1}: n_filters={cfg['n_filters']}, lr={cfg['lr']}")
    t0 = time.time()
    res = train_model(X_sub, y_sub, **cfg)
    elapsed = time.time() - t0

    # Evaluate on test set
    ev = evaluate_model(res, data["X_test"], data["y_test"])

    hp_results.append({
        "config": cfg,
        "test_acc": ev["accuracy"],
        "train_acc": res["history"]["acc"][-1],
        "time": elapsed,
    })
    print(f"  Test acc: {ev['accuracy']:.4f}  ({elapsed:.2f}s)")

# Summary
print("\n" + "="*60)
print(f"{'Config':>30s}  {'Test Acc':>8s}  {'Time':>6s}")
print("-"*60)
for r in sorted(hp_results, key=lambda x: -x["test_acc"]):
    cfg_str = f"f={r['config']['n_filters']}, lr={r['config']['lr']}"
    print(f"{cfg_str:>30s}  {r['test_acc']:>8.4f}  {r['time']:>5.2f}s")

## 7. Cached Feature Extraction

Extract intermediate features from the trained model for downstream analysis (clustering, visualization). This forward pass is expensive on large datasets — caching makes iteration instant.

In [ ]:
@cache_file(CACHE_DIR, backend="pickle", verbose=True)
def extract_features(train_result, X, batch_size=256):
    """Extract penultimate layer features from trained model."""
    model = SimpleCNN(
        n_filters=train_result["n_filters"],
        dropout=train_result["dropout"]
    )
    model.load_state_dict(train_result["state_dict"])
    model.eval()

    # Hook into fc1 output
    features_list = []
    def hook_fn(module, input, output):
        features_list.append(output.detach().cpu())

    handle = model.fc1.register_forward_hook(hook_fn)

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            model(X[i:i+batch_size])

    handle.remove()
    return torch.cat(features_list, dim=0)

t0 = time.time()
features = extract_features(result, data["X_test"])
print(f"Features extracted in {time.time()-t0:.2f}s")
print(f"Feature shape: {features.shape}  (10000 samples x 128 dims)")

In [ ]:
# t-SNE visualization of learned features (not cached — fast enough)
from sklearn.manifold import TSNE

# Use subset for speed
n_vis = 3000
idx = np.random.RandomState(42).choice(len(features), n_vis, replace=False)

t0 = time.time()
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
coords = tsne.fit_transform(features[idx].numpy())
print(f"t-SNE done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(coords[:, 0], coords[:, 1],
                     c=data["y_test"][idx].numpy(), cmap="tab10",
                     s=5, alpha=0.7)
ax.set_title("t-SNE of Learned Features (test set)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label("Digit")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 8. Re-run Demo: Everything from Cache

Simulate re-opening the notebook — all expensive steps are instant.

In [ ]:
print("Re-running the full pipeline (everything cached):\n")
_file_state_cache.clear()

steps = [
    ("Load MNIST",       lambda: load_mnist(str(DATA_DIR), flatten=False)),
    ("Train model",      lambda: train_model(data["X_train"], data["y_train"],
                                              n_filters=32, dropout=0.25, lr=1e-3, epochs=3)),
    ("Evaluate",         lambda: evaluate_model(result, data["X_test"], data["y_test"])),
    ("Extract features", lambda: extract_features(result, data["X_test"])),
]

step_times = []
for name, fn in steps:
    t0 = time.time()
    fn()
    elapsed = time.time() - t0
    step_times.append((name, elapsed))
    print(f"  {name:20s} {elapsed:.4f}s")

total = sum(t for _, t in step_times)
print(f"\nTotal (from cache): {total:.3f}s")

## 9. Speed Comparison: First Execution vs Cache Hit

In [ ]:
# Collect first-execution times by running from scratch
fresh_start()
_file_state_cache.clear()

first_times = []
step_names = ["Load MNIST", "Train model", "Evaluate", "Extract\nfeatures"]

# Load
t0 = time.time()
data2 = load_mnist(str(DATA_DIR), flatten=False)
first_times.append(time.time() - t0)

# Train
t0 = time.time()
result2 = train_model(data2["X_train"], data2["y_train"],
                      n_filters=32, dropout=0.25, lr=1e-3, epochs=3)
first_times.append(time.time() - t0)

# Evaluate
t0 = time.time()
metrics2 = evaluate_model(result2, data2["X_test"], data2["y_test"])
first_times.append(time.time() - t0)

# Features
t0 = time.time()
feat2 = extract_features(result2, data2["X_test"])
first_times.append(time.time() - t0)

# Cache hits
_file_state_cache.clear()
cache_times = []
for fn in [
    lambda: load_mnist(str(DATA_DIR), flatten=False),
    lambda: train_model(data2["X_train"], data2["y_train"],
                        n_filters=32, dropout=0.25, lr=1e-3, epochs=3),
    lambda: evaluate_model(result2, data2["X_test"], data2["y_test"]),
    lambda: extract_features(result2, data2["X_test"]),
]:
    t0 = time.time()
    fn()
    cache_times.append(time.time() - t0)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(step_names))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, first_times, width, label="First execution", color="#e74c3c")
ax.bar(x + width/2, cache_times, width, label="Cache hit", color="#2ecc71")
ax.set_ylabel("Time (seconds)")
ax.set_title("PyTorch Pipeline: First Run vs Cache Hit")
ax.set_xticks(x)
ax.set_xticklabels(step_names)
ax.legend()
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3)

# Speedup
speedups = [f / max(c, 1e-6) for f, c in zip(first_times, cache_times)]
ax2 = axes[1]
ax2.bar(x, speedups, color="#3498db", width=0.5)
ax2.set_ylabel("Speedup factor (x)")
ax2.set_title("Cache Speedup per Step")
ax2.set_xticks(x)
ax2.set_xticklabels(step_names)
ax2.grid(axis="y", alpha=0.3)
for i, s in enumerate(speedups):
    ax2.text(i, s + max(speedups) * 0.02, f"{s:.0f}x",
             ha="center", fontsize=11, fontweight="bold")

fig.tight_layout()
plt.show()

total_first = sum(first_times)
total_cached = sum(cache_times)
print(f"\nTotal: {total_first:.1f}s (first) vs {total_cached:.2f}s (cached) = {total_first/total_cached:.0f}x speedup")

## 10. Inspect the Cache

In [ ]:
stats = cache_stats(CACHE_DIR)
print("Cache statistics:")
print(f"  Entries:    {stats['n_entries']}")
print(f"  Total size: {stats['total_size_mb']:.1f} MB")

print(f"\nCached files:")
for f in sorted(CACHE_DIR.glob("*.pkl")):
    size_kb = f.stat().st_size / 1024
    fname = f.stem.split(".")[0]
    print(f"  {fname:25s}  {size_kb:8.1f} KB")

nodes = cache_tree_nodes()
print(f"\nDependency graph: {len(nodes)} nodes")
for nid, node in nodes.items():
    parents = len(node.get('parents', []))
    children = len(node.get('children', []))
    print(f"  {node['fname']:25s}  parents={parents}  children={children}")

## Cleanup

In [ ]:
# shutil.rmtree(CACHE_DIR)
# shutil.rmtree(DATA_DIR)
# print("Cache and data cleared.")

logging.getLogger().setLevel(logging.WARNING)
print(f"Cache preserved at: {CACHE_DIR.resolve()}")